## *Problem Statement*

The Meteorite Landings dataset contains information about meteorites,
including their mass, year of landing, geographic coordinates and
classification-related attributes.

The objective of this clustering study is to identify meaningful groups
of meteorite records based on their numerical characteristics and
geographic information, without using predefined target labels.

The clustering analysis compares the required K-Means and
Hierarchical/Agglomerative Clustering algorithms using standard
internal clustering evaluation metrics.

### *Clustering Objective*

The clustering task aims to:

- Explore the structure and characteristics of the Meteorite Landings
  dataset.
- Identify meaningful numerical features for clustering.
- Engineer suitable features from meteorite mass and geographic
  coordinates.
- Group meteorite records using the required clustering algorithms.
- Evaluate the quality of the generated clusters using Silhouette Score,
  Davies-Bouldin Index and Calinski-Harabasz Index.
- Visualize the resulting clusters using dimensionality-reduction
  techniques.

## *Import Required Libraries*

### *Import Python Libraries*

In [21]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering
)

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from sklearn.decomposition import PCA

from sklearn.manifold import TSNE

from scipy.cluster.hierarchy import (
    dendrogram,
    linkage
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

### *Library Purpose*

Pandas and NumPy are used for data manipulation and numerical
operations.

Matplotlib and Seaborn are used for exploratory data visualisation.

Scikit-learn is used for feature scaling, clustering, dimensionality
reduction and cluster evaluation.

SciPy is used to generate the hierarchical clustering dendrogram.

A random state of 42 is used wherever applicable to ensure
reproducibility.

## *Dataset Loading*

### *Load Raw Dataset*

In [22]:
!git clone https://github.com/manaswiniyoshitha2006-netizen/ML-Evaluation.git

fatal: destination path 'ML-Evaluation' already exists and is not an empty directory.


In [23]:
df_raw = pd.read_csv("/content/ML-Evaluation/data/Meteorite_Landings.csv")
df = df_raw.copy()
print("Dataset loaded successfully.")

Dataset loaded successfully.


### *Dataset Dimensions*

In [24]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 45716
Number of columns: 10


### *First Five Records*

In [25]:
df.head()

,name,id,nametype,recclass,mass (g),fall,year,reclat,reclong,GeoLocation
0,Aachen,1,Valid,L5,21.0,Fell,1880.0,50.77500,6.08333,"(50.775, 6.08333)"
1,Aarhus,2,Valid,H6,720.0,Fell,1951.0,56.18333,10.23333,"(56.18333, 10.23333)"
2,Abee,6,Valid,EH4,107000.0,Fell,1952.0,54.21667,-113.00000,"(54.21667, -113.0)"
3,Acapulco,10,Valid,Acapulcoite,1914.0,Fell,1976.0,16.88333,-99.90000,"(16.88333, -99.9)"
4,Achiras,370,Valid,L6,780.0,Fell,1902.0,-33.16667,-64.95000,"(-33.16667, -64.95)"


### *Last Five Records*

In [26]:
df.tail()

,name,id,nametype,recclass,mass (g),fall,year,reclat,reclong,GeoLocation
45711,Zillah 002,31356,Valid,Eucrite,172.0,Found,1990.0,29.03700,17.01850,"(29.037, 17.0185)"
45712,Zinder,30409,Valid,"Pallasite, ungrouped",46.0,Found,1999.0,13.78333,8.96667,"(13.78333, 8.96667)"
45713,Zlin,30410,Valid,H4,3.3,Found,1939.0,49.25000,17.66667,"(49.25, 17.66667)"
45714,Zubkovsky,31357,Valid,L6,2167.0,Found,2003.0,49.78917,41.50460,"(49.78917, 41.5046)"
45715,Zulu Queen,30414,Valid,L3.7,200.0,Found,1976.0,33.98333,-115.68333,"(33.98333, -115.68333)"


### *Column Names*

In [27]:
df.columns.tolist()

['name',
 'id',
 'nametype',
 'recclass',
 'mass (g)',
 'fall',
 'year',
 'reclat',
 'reclong',
 'GeoLocation']

## *Dataset Audit*

### *Dataset Information*

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45716 entries, 0 to 45715
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         45716 non-null  object 
 1   id           45716 non-null  int64  
 2   nametype     45716 non-null  object 
 3   recclass     45716 non-null  object 
 4   mass (g)     45585 non-null  float64
 5   fall         45716 non-null  object 
 6   year         45425 non-null  float64
 7   reclat       38401 non-null  float64
 8   reclong      38401 non-null  float64
 9   GeoLocation  38401 non-null  object 
dtypes: float64(4), int64(1), object(5)
memory usage: 3.5+ MB


### *Data Types*

In [29]:
df.dtypes

,0
name,object
id,int64
nametype,object
recclass,object
mass (g),float64
fall,object
year,float64
reclat,float64
reclong,float64
GeoLocation,object


### *Numerical Descriptive Statistics*

In [30]:
df.describe()

,id,mass (g),year,reclat,reclong
count,45716.000000,4.558500e+04,45425.000000,38401.000000,38401.000000
mean,26889.735104,1.327808e+04,1991.828817,-39.122580,61.074319
std,16860.683030,5.749889e+05,25.052766,46.378511,80.647298
min,1.000000,0.000000e+00,860.000000,-87.366670,-165.433330
25%,12688.750000,7.200000e+00,1987.000000,-76.714240,0.000000
50%,24261.500000,3.260000e+01,1998.000000,-71.500000,35.666670
75%,40656.750000,2.026000e+02,2003.000000,0.000000,157.166670
max,57458.000000,6.000000e+07,2101.000000,81.166670,354.473330


### *Categorical Descriptive Statistics*

In [31]:
df.describe(include="object")

,name,nametype,recclass,fall,GeoLocation
count,45716,45716,45716,45716,38401
unique,45716,2,466,2,17100
top,Zulu Queen,Valid,L6,Found,"(0.0, 0.0)"
freq,1,45641,8285,44609,6214


### *Numerical and Categorical Feature Identification*

In [32]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = df.select_dtypes(
    include="object"
).columns.tolist()

print("Numerical features:")
print(numeric_columns)

print("\nCategorical features:")
print(categorical_columns)

Numerical features:
['id', 'mass (g)', 'year', 'reclat', 'reclong']

Categorical features:
['name', 'nametype', 'recclass', 'fall', 'GeoLocation']


## *Missing Value Analysis*

### *Missing Value Count*

In [33]:
missing_count = df.isnull().sum()

missing_count.sort_values(
    ascending=False
)

,0
GeoLocation,7315
reclong,7315
reclat,7315
year,291
mass (g),131
name,0
id,0
fall,0
nametype,0
recclass,0


### *Missing Value Percentage*

In [34]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage":
        (df.isnull().sum() / len(df)) * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
)

missing_summary

,Missing Count,Missing Percentage
GeoLocation,7315,16.000962
reclong,7315,16.000962
reclat,7315,16.000962
year,291,0.636539
mass (g),131,0.286552
name,0,0.000000
id,0,0.000000
fall,0,0.000000
nametype,0,0.000000
recclass,0,0.000000


### *Missing Value Interpretation*

Missing values are examined before clustering because the clustering
algorithms require complete numerical input features.

The missing values in the variables relevant to clustering will be
handled during the data-cleaning and feature-engineering stage.

## *Duplicate Analysis*

### *Duplicate Row Check*

In [35]:
duplicate_rows = df.duplicated().sum()

print(
    "Number of duplicate rows:",
    duplicate_rows
)

Number of duplicate rows: 0


### *Duplicate ID Check*

In [36]:
duplicate_ids = df["id"].duplicated().sum()

print(
    "Number of duplicate IDs:",
    duplicate_ids
)

Number of duplicate IDs: 0


### *Duplicate Treatment Decision*

Duplicate records are examined because repeated observations can affect
the distribution of the dataset and influence clustering results.

The final treatment decision will be based on the observed duplicate
records and their validity.

## *Data Consistency and Logical Validation*

### *Meteorite Classification Consistency*

In [37]:
print("Unique nametype values:")
print(df["nametype"].value_counts(dropna=False))

print("\nNumber of unique recclass values:")
print(df["recclass"].nunique())

Unique nametype values:
nametype
Valid     45641
Relict       75
Name: count, dtype: int64

Number of unique recclass values:
466


### *Latitude and Longitude Validation*

In [38]:
invalid_latitude = (
    (df["reclat"] < -90) |
    (df["reclat"] > 90)
).sum()

invalid_longitude = (
    (df["reclong"] < -180) |
    (df["reclong"] > 180)
).sum()

print(
    "Invalid latitude values:",
    invalid_latitude
)

print(
    "Invalid longitude values:",
    invalid_longitude
)

Invalid latitude values: 0
Invalid longitude values: 1


### *Meteorite Mass Validation*

In [39]:
negative_mass = (
    df["mass (g)"] < 0
).sum()

zero_mass = (
    df["mass (g)"] == 0
).sum()

print("Negative mass values:", negative_mass)
print("Zero mass values:", zero_mass)

Negative mass values: 0
Zero mass values: 19


### *Year Validation*

In [40]:
print(
    "Minimum year:",
    df["year"].min()
)

print(
    "Maximum year:",
    df["year"].max()
)

Minimum year: 860.0
Maximum year: 2101.0
